# EXEMPLO — Pipeline RAG Mínimo Viável

**Curso:** MBA RAG & CAG Aplicados a Direito e Segurança Pública  
**Aula:** 1 — Fundamentos  
**Tipo:** Exemplo demonstrativo (não é laboratório avaliado)  
**Ambiente:** Jupyter Local no VS Code

---

## O que este exemplo demonstra

Este notebook conecta todos os componentes aprendidos nos Labs 1-4:

```
PASSO 1: Corpus juridico (JSON)
    |
    v
PASSO 2: Embeddings (BGE-M3 via sentence-transformers)
    |
    v
PASSO 3: Indice vetorial (FAISS local)
    |
    v
PASSO 4: Retrieval semantico (busca por similaridade coseno)
    |
    v
PASSO 5: Geracao com LLM (Ollama + llama3.2:3b, API OpenAI-compat)
    |
    v
PASSO 6: Observabilidade (LangFuse — trace completo)
```

Este e o **Pipeline RAG minimo** que sera expandido ao longo de todas as 12 aulas.

---

## Pre-requisitos
- Labs 1 a 4 concluidos
- Ollama rodando com `llama3.2:3b` e `nomic-embed-text` instalados
- (Opcional) LangFuse configurado no `.env`

In [ ]:
# PASSO 0: Verificar ambiente
import sys, torch, requests

print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {DEVICE}')

# Verificar Ollama
OLLAMA_BASE_URL = 'http://localhost:11434'
try:
    r = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=5)
    modelos = [m['name'] for m in r.json().get('models', [])]
    print(f'Ollama: ONLINE — modelos: {modelos}')
    OLLAMA_OK = True
except Exception as e:
    print(f'Ollama: OFFLINE ({e})')
    print('  Execute: ollama serve')
    OLLAMA_OK = False


In [ ]:
# PASSO 0.1: Instalar dependencias
%pip install sentence-transformers FlagEmbedding faiss-cpu openai langfuse python-dotenv --quiet
print('Dependencias prontas!')


In [ ]:
# PASSO 0.2: Imports e configuracao
import json
import numpy as np
import requests
import os
import time
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import faiss
from openai import OpenAI

# Carregar .env se disponivel
try:
    from dotenv import load_dotenv
    for env_path in [Path.home() / 'mba-rag' / '.env', Path('.env'), Path('..') / '.env']:
        if env_path.exists():
            load_dotenv(env_path)
            print(f'Configuracoes carregadas de: {env_path}')
            break
except ImportError:
    pass

# Configuracoes de LLM e Embedding
OLLAMA_BASE_URL  = os.environ.get('OLLAMA_BASE_URL', 'http://localhost:11434')
OLLAMA_LLM_MODEL = os.environ.get('OLLAMA_LLM_MODEL', 'llama3.2:3b')
EMBED_MODEL_OLLAMA = os.environ.get('OLLAMA_EMBED_MODEL', 'nomic-embed-text')

# Cliente OpenAI-compativel apontando para Ollama
client_llm = OpenAI(
    base_url=f'{OLLAMA_BASE_URL}/v1',
    api_key='ollama'
)

print(f'LLM: {OLLAMA_LLM_MODEL} via {OLLAMA_BASE_URL}')
print(f'Embedding: {EMBED_MODEL_OLLAMA} via Ollama')
print('Imports prontos!')


## PASSO 1 — Carregar o Corpus Jurídico

**O que faz:** Carrega os 10 documentos jurídicos fictícios do dataset da aula.  
**Por que:** Em producao, este seria o banco de dados de acórdãos, BOs ou laudos periciais.

In [ ]:
# PASSO 1: Carregar corpus juridico
# Tenta carregar do arquivo local primeiro, fallback para corpus embutido
from pathlib import Path

possiveis_paths = [
    Path('../datasets/corpus_juridico_aula1.json'),
    Path('../../datasets/corpus_juridico_aula1.json'),
    Path('./datasets/corpus_juridico_aula1.json'),
]

corpus = None
for corpus_path in possiveis_paths:
    if corpus_path.exists():
        with open(corpus_path, encoding='utf-8') as f:
            corpus = json.load(f)
        print(f'Corpus carregado de: {corpus_path.resolve()}')
        break

if corpus is None:
    # Corpus embutido para demo
    print('Arquivo nao encontrado — usando corpus embutido para demo')
    corpus = [
        {'id': '1', 'titulo': 'Peculato doloso - servidor publico',
         'texto': 'Servidor publico desviou R$2.5M do fundo do SUS. Condenado pelo art. 312 CP a 8 anos de reclusao.'},
        {'id': '2', 'titulo': 'Habeas corpus - prisao preventiva excessiva',
         'texto': 'Prisao preventiva sem fundamentacao adequada. HC concedido por excesso de prazo. Art. 316 CPP.'},
        {'id': '3', 'titulo': 'Furto qualificado mediante escalada',
         'texto': 'Reu subtraiu bens mediante escalada. Furto qualificado art. 155 par. 4o CP. Pena 2 a 8 anos.'},
        {'id': '4', 'titulo': 'Homicidio qualificado - motivo futl',
         'texto': 'Vitima assassinada por motivo futl apos discussao. Homicidio qualificado art. 121 par. 2o CP.'},
        {'id': '5', 'titulo': 'Trafico de drogas - flagrante',
         'texto': 'Preso em flagrante com 500g de cocaina. Trafico art. 33 Lei 11.343/06. Pena 5 a 15 anos.'},
        {'id': '6', 'titulo': 'Corrupcao passiva - fiscal tributario',
         'texto': 'Fiscal exigiu vantagem para arquivar autuacao. Corrupcao passiva art. 317 CP. Demissao e reclusao.'},
        {'id': '7', 'titulo': 'Legítima defesa - excludente de ilicitude',
         'texto': 'Reu reagiu a agressao injusta com moderacao. Legitima defesa reconhecida. Art. 25 CP. Absolvido.'},
        {'id': '8', 'titulo': 'Estelionato - fraude bancaria digital',
         'texto': 'Reu usou phishing para roubar credenciais e desviar R$80k. Estelionato art. 171 CP.'},
        {'id': '9', 'titulo': 'Violencia domestica - Lei Maria da Penha',
         'texto': 'Agressor descumpriu medida protetiva. Prisao em flagrante. Lei 11.340/06 art. 22 e 313 CPP.'},
        {'id': '10', 'titulo': 'Roubo a mao armada - concurso de agentes',
         'texto': 'Dois acusados roubaram agencia bancaria com armas de fogo. Roubo qualificado art. 157 par. 2o CP.'},
    ]

print(f'Corpus: {len(corpus)} documentos')
for doc in corpus[:3]:
    print(f'  [{doc["id"]}] {doc["titulo"]}')
print('  ...')


## PASSO 2 — Gerar Embeddings

**Duas estratégias disponíveis:**
- **Ollama** (`nomic-embed-text`): mais simples, API REST, sem gerenciar modelos Python
- **sentence-transformers** (`BAAI/bge-m3`): maior controle, melhor qualidade, ~570MB de download

Este exemplo usa **ambas** para demonstração. Em produção, escolha uma.

In [ ]:
# PASSO 2: Gerar embeddings do corpus

# ─── Estrategia A: Embedding via Ollama (padrao do curso) ────
def gerar_embedding_ollama(texto: str, modelo: str = EMBED_MODEL_OLLAMA) -> list:
    """Gera embedding via Ollama API REST."""
    r = requests.post(
        f'{OLLAMA_BASE_URL}/api/embeddings',
        json={'model': modelo, 'prompt': texto},
        timeout=30
    )
    r.raise_for_status()
    return r.json()['embedding']

def gerar_embeddings_corpus_ollama(corpus: list) -> np.ndarray:
    """Gera embeddings para todos os documentos via Ollama."""
    embeddings = []
    for doc in corpus:
        texto = f"{doc['titulo']}. {doc['texto']}"
        emb = gerar_embedding_ollama(texto)
        embeddings.append(emb)
    return np.array(embeddings, dtype='float32')

# ─── Estrategia B: sentence-transformers (BGE-M3) ────────────
def gerar_embeddings_corpus_sbert(corpus: list, encoder) -> np.ndarray:
    """Gera embeddings usando sentence-transformers."""
    textos = [f"{doc['titulo']}. {doc['texto']}" for doc in corpus]
    return encoder.encode(textos, convert_to_numpy=True, show_progress_bar=True)

# ─── Execucao (usa Ollama se disponivel, fallback para BGE-M3) ─
print('Gerando embeddings do corpus...')

# Verifica se nomic-embed-text esta disponivel no Ollama
try:
    modelos_ollama = [m['name'] for m in requests.get(
        f'{OLLAMA_BASE_URL}/api/tags', timeout=3).json().get('models', [])]
    usar_ollama_embed = any('nomic' in m or 'embed' in m for m in modelos_ollama)
except Exception:
    usar_ollama_embed = False

if usar_ollama_embed:
    print(f'Usando Ollama ({EMBED_MODEL_OLLAMA})...')
    corpus_embeddings = gerar_embeddings_corpus_ollama(corpus)
    encoder_nome = f'Ollama/{EMBED_MODEL_OLLAMA}'
else:
    print('nomic-embed-text nao encontrado — carregando BGE-M3 via sentence-transformers...')
    print('(Download ~570MB na primeira vez)')
    encoder = SentenceTransformer('BAAI/bge-m3', device=DEVICE)
    corpus_embeddings = gerar_embeddings_corpus_sbert(corpus, encoder)
    encoder_nome = 'sentence-transformers/bge-m3'

print(f'Embeddings gerados: {corpus_embeddings.shape}')
print(f'  Documentos: {corpus_embeddings.shape[0]}')
print(f'  Dimensoes:  {corpus_embeddings.shape[1]}')
print(f'  Encoder:    {encoder_nome}')


## PASSO 3 — Indexar no FAISS

**O que faz:** Cria um índice vetorial FAISS com os embeddings do corpus.  
**Por que:** FAISS permite busca por similaridade em milissegundos mesmo com milhões de vetores.

In [ ]:
# PASSO 3: Indexar no FAISS
import faiss

DIM = corpus_embeddings.shape[1]

# IndexFlatIP = produto interno (equivale a similaridade coseno com vetores normalizados)
index = faiss.IndexFlatIP(DIM)

# Normalizar vetores para similaridade coseno
embeddings_norm = corpus_embeddings.copy()
faiss.normalize_L2(embeddings_norm)
index.add(embeddings_norm)

print(f'Indice FAISS criado')
print(f'  Tipo: IndexFlatIP (produto interno, L2-normalizado)')
print(f'  Vetores: {index.ntotal}')
print(f'  Dimensoes: {DIM}')


## PASSO 4 — Retrieval Semantico

**O que faz:** Recebe uma query em linguagem natural, encodifica, busca no FAISS e retorna os K documentos mais similares.

In [ ]:
# PASSO 4: Funcao de retrieval semantico

def busca_semantica(query: str, top_k: int = 3) -> list:
    """
    Busca semantica no indice FAISS.
    1. Encodifica a query com o mesmo encoder do corpus
    2. Normaliza o vetor
    3. Busca os top_k mais proximos no FAISS
    4. Retorna os documentos com scores
    """
    # Gerar embedding da query
    if usar_ollama_embed:
        query_emb = np.array([gerar_embedding_ollama(query)], dtype='float32')
    else:
        query_emb = encoder.encode([query], convert_to_numpy=True)

    # Normalizar
    faiss.normalize_L2(query_emb)

    # Buscar no FAISS
    scores, indices = index.search(query_emb, top_k)

    # Montar resultado
    resultados = []
    for score, idx in zip(scores[0], indices[0]):
        if idx >= 0:
            doc = corpus[idx].copy()
            doc['score'] = float(score)
            resultados.append(doc)

    return resultados

# Teste de retrieval
QUERY_TESTE = 'crime de servidor publico que desviou dinheiro do governo'

print(f'Query: "{QUERY_TESTE}"')
print('=' * 60)

docs_recuperados = busca_semantica(QUERY_TESTE, top_k=3)

for i, doc in enumerate(docs_recuperados):
    print(f'[{i+1}] Score: {doc["score"]:.3f} — {doc["titulo"]}')
    print(f'      {doc["texto"][:100]}...')
    print()


## PASSO 5 — Geração com LLM (Ollama)

**O que faz:** Monta o prompt RAG com os documentos recuperados e chama o Llama via Ollama.  
**Por que:** O LLM usa os documentos como contexto para gerar uma resposta fundamentada — isso é RAG.

In [ ]:
# PASSO 5: Geracao com LLM + RAG

# Monta o contexto com os documentos recuperados
contexto = '\n\n'.join([
    f'[Documento {i+1}] Titulo: {doc["titulo"]}\nConteudo: {doc["texto"]}'
    for i, doc in enumerate(docs_recuperados)
])

SYSTEM_RAG = (
    'Voce e um assistente juridico especializado em direito penal brasileiro. '
    'Responda APENAS com base nos documentos fornecidos abaixo. '
    'Se a informacao nao estiver nos documentos, diga explicitamente. '
    'Nunca invente artigos, precedentes ou fatos nao presentes nos documentos.'
)

USER_RAG = f"""DOCUMENTOS RECUPERADOS:
{contexto}

PERGUNTA: {QUERY_TESTE}

Resposta fundamentada nos documentos:"""

print(f'Modelo LLM: {OLLAMA_LLM_MODEL}')
print('Gerando resposta RAG...')
print()

inicio = time.time()

resposta_rag = client_llm.chat.completions.create(
    model=OLLAMA_LLM_MODEL,
    messages=[
        {'role': 'system', 'content': SYSTEM_RAG},
        {'role': 'user',   'content': USER_RAG}
    ],
    temperature=0.2,
    max_tokens=400,
)

tempo = time.time() - inicio
texto_resposta = resposta_rag.choices[0].message.content

print('RESPOSTA DO PIPELINE RAG:')
print('-' * 60)
print(texto_resposta)
print('-' * 60)
print(f'\nMetricas: {resposta_rag.usage.total_tokens} tokens em {tempo:.1f}s')


## PASSO 6 — Registrar no LangFuse (Observabilidade)

**O que faz:** Registra o trace completo do pipeline RAG no LangFuse.  
**Por que:** Permite auditoria da qualidade das respostas e monitoramento em producao.

> **Opcional:** Se nao tiver LangFuse configurado, esta celula exibirá um aviso e continuará.

In [ ]:
# PASSO 6: Observabilidade com LangFuse
import uuid

LANGFUSE_PUBLIC_KEY = os.environ.get('LANGFUSE_PUBLIC_KEY', '')
LANGFUSE_SECRET_KEY = os.environ.get('LANGFUSE_SECRET_KEY', '')
LANGFUSE_HOST       = os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com')

langfuse_ok = (
    LANGFUSE_PUBLIC_KEY
    and not LANGFUSE_PUBLIC_KEY.endswith('_AQUI')
    and len(LANGFUSE_PUBLIC_KEY) > 10
)

if langfuse_ok:
    from langfuse import Langfuse

    langfuse = Langfuse(
        public_key=LANGFUSE_PUBLIC_KEY,
        secret_key=LANGFUSE_SECRET_KEY,
        host=LANGFUSE_HOST
    )

    trace = langfuse.trace(
        name='pipeline-rag-minimo',
        session_id=str(uuid.uuid4()),
        metadata={
            'modulo': 'MBA RAG - Aula 1',
            'encoder': encoder_nome if 'encoder_nome' in dir() else EMBED_MODEL_OLLAMA,
            'llm_model': OLLAMA_LLM_MODEL,
            'top_k': 3,
        },
        tags=['aula1', 'exemplo', 'pipeline-minimo']
    )

    # Retrieval span
    retrieval_span = trace.span(name='faiss-retrieval', metadata={'query': QUERY_TESTE, 'top_k': 3})
    retrieval_span.end(output=[{'titulo': d['titulo'], 'score': d['score']} for d in docs_recuperados])

    # Generation
    gen = trace.generation(
        name='ollama-generation',
        model=OLLAMA_LLM_MODEL,
        model_parameters={'temperature': 0.2, 'max_tokens': 400},
        input=USER_RAG[:500],
        output=texto_resposta,
        usage={
            'input': resposta_rag.usage.prompt_tokens,
            'output': resposta_rag.usage.completion_tokens,
        }
    )
    gen.end()

    langfuse.flush()
    print(f'Trace registrado no LangFuse!')
    print(f'  Trace ID: {trace.id}')
    print(f'  Dashboard: {LANGFUSE_HOST}')
else:
    print('LangFuse nao configurado (opcional).')
    print('Para ativar: adicione LANGFUSE_PUBLIC_KEY e LANGFUSE_SECRET_KEY ao .env')


## Resumo do Pipeline RAG Minimo

Voce acabou de executar um pipeline RAG completo com:

| Componente | Implementacao | Proximo upgrade |
|-----------|--------------|----------------|
| Corpus | JSON local / embutido | Docling (Aula 4) |
| Embedding | Ollama / BGE-M3 | Embedding fine-tuned (Aula 6) |
| Indice | FAISS local | OpenSearch (Lab 1 + Aulas 5-6) |
| Retrieval | Similaridade coseno | HyDE, Multi-query (Aula 6) |
| LLM | Ollama llama3.2:3b | Reranking (Aula 7) |
| Observabilidade | LangFuse | RAGAs metrics (Aula 10) |

**Este pipeline sera expandido aula a aula ao longo do curso.**